# LambdaG Visualization — Certamina septem fratrum Machabeorum vs. Comparison Candidates

This notebook generates the local token-level LambdaG visualizations for the hold-out work `X4_certamina`.

The poem remains excluded from training and calibration. For each candidate in the shared hexameter test corpus, the visualization compares the complete masked *Certamina septem fratrum Machabeorum* with up to 2000 tokens of the candidate and 20,000 tokens from the reference population.

`X3_machabeis` is also reintroduced here as an **analysis-only candidate**, exactly as in the comparative authorship-verification notebook. It remains excluded from training and calibration and is never used in the reference population for the other candidates.

The HTML files are saved locally in the poem-specific folder.


## Stage 1: Setup

The shared corpora and processed data are stored one directory above this notebook. HTML outputs remain in the current *Certamina* folder.


In [ ]:
from pathlib import Path
import numpy as np

from sentence_corpora import TwoLevelCorpus
from lambdag.visualize import lambdaG_visualize

# ---------------------------------------------------------
# Shared files produced by the general hexameter pipeline
# ---------------------------------------------------------
GENERAL_DIR = Path("..").resolve()

TEST_CORPUS_PATH = GENERAL_DIR / "test_corpus.pkl"
CERTAMINA_CORPUS_PATH = GENERAL_DIR / "certamina_corpus.pkl"
MACHABEIS_CORPUS_PATH = GENERAL_DIR / "machabeis_corpus.pkl"
DATA_PROCESSED_DIR = GENERAL_DIR / "data_processed"

# Local output directory
OUTPUT_DIR = Path("html_certamina")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [
    TEST_CORPUS_PATH,
    CERTAMINA_CORPUS_PATH,
    MACHABEIS_CORPUS_PATH,
    DATA_PROCESSED_DIR,
]:
    if not path.exists():
        raise FileNotFoundError(f"Required path not found: {path}")

test_corpus = TwoLevelCorpus.from_pickle(str(TEST_CORPUS_PATH))
holdout_corpus = TwoLevelCorpus.from_pickle(str(CERTAMINA_CORPUS_PATH))
machabeis_corpus = TwoLevelCorpus.from_pickle(str(MACHABEIS_CORPUS_PATH))

# Sanity checks: both uncertain texts must remain outside the shared test corpus
test_authors = set(test_corpus.get_unique_values("author"))
assert "X4" not in test_authors, "X4_certamina must not be present in test_corpus"
assert "X3" not in test_authors, "X3_machabeis must not be present in test_corpus"

assert set(holdout_corpus.get_unique_values("author")) == {"X4"}
assert set(machabeis_corpus.get_unique_values("author")) == {"X3"}

print(f"General analysis directory: {GENERAL_DIR}")
print(f"HTML output directory: {OUTPUT_DIR.resolve()}")
print(f"test_corpus: {len(test_corpus)} sentences")
print(
    f"Certamina: {len(holdout_corpus)} sentences, "
    f"{sum(len(s.text.split()) for s in holdout_corpus)} tokens"
)
print(
    f"De Machabeis: {len(machabeis_corpus)} sentences, "
    f"{sum(len(s.text.split()) for s in machabeis_corpus)} tokens"
)
print(f"Authors in test_corpus: {sorted(test_corpus.get_unique_values('author'))}")


## Stage 2: Visualization per Candidate

For each comparison candidate:

1. use up to 2000 tokens of candidate text;
2. use 20,000 reference tokens;
3. use the complete masked *Certamina septem fratrum Machabeorum* as questioned text;
4. align the masked sentences with the unmasked `.quesplit.txt` version when available;
5. save masked and unmasked HTML visualizations in `html_certamina/`.

For the ordinary background candidates, the reference population consists of the other authors in `test_corpus`. For `X3` (*De Machabeis*), the candidate text comes from its hold-out corpus while the reference population remains the known-authorship `test_corpus`.


In [ ]:
# Background candidates plus X3_machabeis as analysis-only candidate
background_candidates = sorted(test_corpus.get_unique_values("author"))
candidates = background_candidates + ["X3"]

print(
    f"Found {len(background_candidates)} background candidates + "
    "X3 (De Machabeis)"
)

rng = np.random.default_rng(seed=42)

# Complete questioned text: Certamina
q_sentences = [s.text.split() for s in holdout_corpus]
print(f"Questioned sentences (X4_certamina): {len(q_sentences)}")

# Load unmasked Certamina sentences once
unmasked_sents = None
quesplit_files = list(
    DATA_PROCESSED_DIR.glob("*certamina_sentences.quesplit.txt")
)

if quesplit_files:
    quesplit_file = quesplit_files[0]
    with quesplit_file.open("r", encoding="utf-8") as f:
        unmasked_sents = [line.strip().split() for line in f if line.strip()]

    print(
        f"Unmasked sentences loaded from {quesplit_file.name}: "
        f"{len(unmasked_sents)}"
    )

    if len(q_sentences) != len(unmasked_sents):
        print(
            "WARNING: masked/unmasked sentence count mismatch "
            f"({len(q_sentences)} vs {len(unmasked_sents)}); "
            "unmasked visualization will be skipped."
        )
        unmasked_sents = None
else:
    print(
        "No Certamina quesplit file found for pattern: "
        "*certamina_sentences.quesplit.txt"
    )

for candidate in candidates:
    print(f"\n{'=' * 60}")
    print(f"Processing candidate: {candidate}")
    print(f"{'=' * 60}")

    # -----------------------------------------------------
    # Candidate corpus
    # -----------------------------------------------------
    if candidate == "X3":
        known_corpus = machabeis_corpus

        # X3 is itself a holdout. When it is the candidate, the
        # known-authorship test corpus remains the reference population.
        ref_corpus = test_corpus
        candidate_label = "X3_machabeis"
    else:
        known_corpus = test_corpus.filter_by_level("author", candidate)

        # For ordinary background candidates, exclude the candidate
        # from the reference population.
        ref_corpus = TwoLevelCorpus(
            [s for s in test_corpus if s.author != candidate],
            levels=test_corpus.levels,
        )
        candidate_label = candidate

    known_total_tokens = sum(len(s.text.split()) for s in known_corpus)
    known_target_tokens = min(2000, known_total_tokens)

    if known_target_tokens == 0:
        print(f"Skipping {candidate}: no usable candidate text")
        continue

    known_sentences_raw, known_breakdown = known_corpus.sample_balanced(
        tokens=known_target_tokens,
        rng=rng,
    )
    known_sentences = [list(s.text.split()) for s in known_sentences_raw]

    print(
        f"Known sample: {sum(len(s) for s in known_sentences)} tokens "
        f"from {len(known_sentences)} sentences"
    )
    print(f"Known breakdown: {known_breakdown}")

    # -----------------------------------------------------
    # Reference corpus: 20,000 tokens
    # -----------------------------------------------------
    ref_total_tokens = sum(len(s.text.split()) for s in ref_corpus)
    ref_target_tokens = min(20000, ref_total_tokens)

    ref_sentences_raw, ref_breakdown = ref_corpus.sample_balanced(
        tokens=ref_target_tokens,
        rng=rng,
    )
    ref_sentences = [list(s.text.split()) for s in ref_sentences_raw]

    print(
        f"Reference sample: {sum(len(s) for s in ref_sentences)} tokens "
        f"from {len(ref_sentences)} sentences"
    )
    print(f"Reference breakdown: {ref_breakdown}")

    # -----------------------------------------------------
    # Masked visualization
    # -----------------------------------------------------
    masked_path = OUTPUT_DIR / f"lambdag_output_{candidate_label}_masked.html"

    result = lambdaG_visualize(
        q_text=q_sentences,
        k_text=known_sentences,
        ref_text=ref_sentences,
        N=8,
        r=30,
        output="html",
        scale="relative",
        negative=True,
        order_by="importance",
        rng=rng,
        print_path=str(masked_path),
    )

    print(f"Masked HTML saved to: {masked_path}")
    print(f"Table shape: {result['table'].shape}")
    print(f"Sentence count: {result['table']['sentence_id'].nunique()}")

    # -----------------------------------------------------
    # Unmasked visualization
    # -----------------------------------------------------
    if unmasked_sents is not None:
        unmasked_path = OUTPUT_DIR / f"lambdag_output_{candidate_label}_unmasked.html"

        result_unmasked = lambdaG_visualize(
            q_text=q_sentences,
            k_text=known_sentences,
            ref_text=ref_sentences,
            N=8,
            r=30,
            output="html",
            scale="relative",
            negative=True,
            order_by="",
            unmasked_sentences=unmasked_sents,
            rng=rng,
            print_path=str(unmasked_path),
        )

        print(f"Unmasked HTML saved to: {unmasked_path}")
    else:
        result_unmasked = None
        print("Skipping unmasked visualization")

    # -----------------------------------------------------
    # Summary statistics
    # -----------------------------------------------------
    table = result["table"]
    sentence_table = table.drop_duplicates("sentence_id")

    print("\nSummary statistics:")
    print(f"  Sentences: {table['sentence_id'].nunique()}")
    print(f"  Tokens: {len(table)}")
    print(f"  Mean sentence λG: {sentence_table['sentence_lambdaG'].mean():.3f}")
    print(f"  Std sentence λG: {sentence_table['sentence_lambdaG'].std():.3f}")
    print(
        f"  Positive tokens: {(table['lambdaG'] > 0).sum()} "
        f"({(table['lambdaG'] > 0).mean() * 100:.1f}%)"
    )
    print(
        f"  Negative tokens: {(table['lambdaG'] < 0).sum()} "
        f"({(table['lambdaG'] < 0).mean() * 100:.1f}%)"
    )


## Stage 3: Optional combined diagnostic

The candidate-specific HTML files generated above are the outputs used for local interpretation. The following cell is retained only as an optional corpus diagnostic and does not enter the attribution results.


In [ ]:
# OPTIONAL DIAGNOSTIC: inspect a balanced 2000-token sample
# from the complete known-authorship test corpus.

rng_combined = np.random.default_rng(seed=42)

test_total_tokens = sum(len(s.text.split()) for s in test_corpus)
sample_target_tokens = min(2000, test_total_tokens)

sample_raw, sample_breakdown = test_corpus.sample_balanced(
    tokens=sample_target_tokens,
    rng=rng_combined,
)

print(
    f"Optional combined sample: "
    f"{sum(len(s.text.split()) for s in sample_raw)} tokens "
    f"from {len(sample_raw)} sentences"
)
print(f"Breakdown: {sample_breakdown}")
